# Q-SHIELD: Quantum AI for Healthcare & Biomedical Diagnostics\n## Drug-Abuse Risk Mapping & Prevention — QAOA Prototype\n\nThis notebook demonstrates a privacy-aware, area-level prevention prototype using synthetic data. It does **not** identify individuals or locate illegal substances.

In [ ]:
import numpy as np\nfrom qiskit.circuit.library import QAOAAnsatz\nfrom qiskit.quantum_info import Operator, Statevector\n

In [ ]:
areas = ['Area A', 'Area B', 'Area C', 'Area D']\nrisks = np.array([0.82, 0.35, 0.91, 0.67])\nbudget = 2\npenalty = 2.0\nlist(zip(areas, risks))

In [ ]:
def cost(bits):\n    x = np.array(bits, dtype=int)\n    return -float(np.dot(risks, x)) + penalty * (int(x.sum()) - budget) ** 2\n\ndim = 2 ** len(areas)\ndiagonal = []\nfor state in range(dim):\n    bits = [(state >> i) & 1 for i in range(len(areas))]\n    diagonal.append(cost(bits))\ncost_operator = Operator(np.diag(diagonal))\nqaoa = QAOAAnsatz(cost_operator=cost_operator, reps=1, flatten=True)\nqaoa

In [ ]:
best = None\nfor gamma in np.linspace(0, 2*np.pi, 9):\n    for beta in np.linspace(0, np.pi, 9):\n        state = Statevector.from_instruction(qaoa.assign_parameters([gamma, beta]))\n        probs = state.probabilities()\n        energy = 0.0\n        for i, p in enumerate(probs):\n            bits = [(i >> j) & 1 for j in range(len(areas))]\n            energy += p * cost(bits)\n        if best is None or energy < best[0]:\n            best = (energy, gamma, beta, probs)\nbest[:3]

In [ ]:
valid = []\nfor i, p in enumerate(best[3]):\n    bits = [(i >> j) & 1 for j in range(len(areas))]\n    if sum(bits) == budget:\n        valid.append((p, bits))\nprobability, bits = max(valid, key=lambda x: x[0])\nselected = [areas[i] for i,b in enumerate(bits) if b]\nprint('Selected prevention priority areas:', selected)\nprint('Bitstring:', ''.join(map(str,bits)))\nprint('QAOA expected energy:', round(best[0], 4))